# FinSight Week 1 — Data Exploration

This notebook walks through fetching, cleaning, and inspecting earnings call transcripts.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from rich import print as rprint
from src.ingest.edgar import TranscriptFetcher
from src.process.store import TranscriptStore
from src.process.cleaner import TranscriptCleaner

## 1. Fetch AAPL transcripts from EDGAR

In [ ]:
fetcher = TranscriptFetcher()
results = fetcher.fetch_by_ticker('AAPL', 2023, 2023)
rprint(f'Fetched [bold]{len(results)}[/bold] documents')
for r in results[:3]:
    rprint(f"  {r['filed_date']}  Q{r['quarter']} {r['year']}  {r['url'][:80]}")

## 2. Clean and chunk a transcript

In [ ]:
cleaner = TranscriptCleaner()

if results:
    raw = results[0]['raw_text']
    cleaned = cleaner.clean(raw)
    chunks  = cleaner.chunk(cleaned, max_tokens=2000)
    sections = cleaner.detect_sections(cleaned)
    rprint(f'Raw chars: {len(raw):,} → Cleaned: {len(cleaned):,}')
    rprint(f'Chunks: {len(chunks)}')
    rprint('Sections detected:', list(sections.keys()))
    rprint('\nFirst chunk preview:')
    print(chunks[0][:500])

## 3. Store and query

In [ ]:
store = TranscriptStore()

for r in results:
    r['cleaned_text'] = cleaner.clean(r['raw_text'])
    saved = store.save(r)
    status = 'saved' if saved else 'duplicate'
    rprint(f"  {r['filed_date']}  {status}")

rprint('\nStore stats:', store.get_stats())

## 4. Retrieve by ticker

In [ ]:
rows = store.get_by_ticker('AAPL')
rprint(f'Stored transcripts for AAPL: {len(rows)}')
for row in rows:
    rprint(f"  id={row['id']}  Q{row['quarter']}/{row['year']}  {row['filed_date']}")